In [1]:
from pyecharts import options as opts
from pyecharts.charts import Tree
import pandas as pd

In [2]:
df = pd.read_excel("../data/AESP_en.xlsx", dtype=str)
df = df.apply(lambda s: s.str.strip()).dropna(how="all")
df.columns = ["Level1", "Level2","Level3", "Level4","Level5", "Level6"]
df

,Level1,Level2,Level3,Level4,Level5,Level6
0,Cognition,Perception,External hallucinations,Acoustic-verbal hallucinations,NaN,NaN
1,Cognition,Perception,External hallucinations,Auditory hallucinations,NaN,NaN
2,Cognition,Perception,External hallucinations,Visual hallucinations,NaN,NaN
3,Cognition,Perception,External hallucinations,Olfactory hallucinations,NaN,NaN
4,Cognition,Perception,External hallucinations,Gustatory hallucinations,NaN,NaN
...,...,...,...,...,...,...
239,Behavior,Physiological functions,Appetite,Decreased,Anorexia,NaN
240,Behavior,Physiological functions,Appetite,Increased,Hyperphagia,NaN
241,Behavior,Physiological functions,Appetite,Altered,Bulimia,NaN
242,Behavior,Physiological functions,Appetite,Altered,Pica,NaN


In [6]:
arbre_dict = {}
for chemin in df.itertuples(index=False):
    noeud = arbre_dict
    for nom in chemin:
        if pd.isna(nom):          # niveau absent : on arrête ce chemin
            break
        noeud = noeud.setdefault(nom, {})

def convertir(d):
    return [{"name": k, "children": convertir(v)} for k, v in d.items()]

racine = {"name": "AESP", "children": convertir(arbre_dict)}


COULEURS = {
    "Behavior": "##65C5CA",
    "Affect": "#f89c74",
    "Cognition": "#f7cf71"
}
DEFAUT = "#CCCCCC"
PAS = 0.3   # éclaircissement

def eclaircir(hex_c, f):
    h = hex_c.lstrip("#")
    rgb = [int(h[i:i+2], 16) for i in (0, 2, 4)]
    return "#%02x%02x%02x" % tuple(round(v + (255 - v) * f) for v in rgb)

def colorier(noeud, c, prof=0):
    cc = eclaircir(c, min(PAS * prof, 0.7))
    noeud["itemStyle"] = {"color": cc}
    noeud["lineStyle"] = {"color": cc, "width": 2}
    for e in noeud.get("children", []):
        colorier(e, c, prof + 1)

for branche in racine["children"]:
    colorier(branche, COULEURS.get(branche["name"], DEFAUT))

for branche in racine["children"]:
    branche["lineStyle"] = {**branche["lineStyle"], "curveness": 0}

arbre = Tree(init_opts=opts.InitOpts(width="800px", height="800px")).add("", [racine], layout="radial",initial_tree_depth=3,label_opts=opts.LabelOpts(font_size=10))
arbre.render("../html/tree_AESP.html")
arbre.render_notebook()